# Knowledge Distillation: Learning from Large Models

## Learning Objectives
1. Understand why soft targets contain more information than hard labels
2. Implement temperature-scaled distillation loss
3. Train small student models from large teacher models
4. Analyze accuracy recovery and compression ratios

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List
import time

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)

print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Level 1: Understanding Soft Targets

Compare hard labels (one-hot) vs soft targets (teacher probabilities).

In [ ]:
# Create a simple example showing difference between hard and soft targets

class SimpleClassifier(nn.Module):
    """Simple neural network for classification."""
    
    def __init__(self, input_size: int = 10, hidden_size: int = 64, num_classes: int = 4):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.relu(self.fc1(x))
        logits = self.fc2(x)
        return logits

# Create synthetic data
torch.manual_seed(0)
batch_size = 32
num_classes = 4

X = torch.randn(batch_size, 10)
y_true = torch.randint(0, num_classes, (batch_size,))

# Create a "teacher" model (well-trained)
teacher = SimpleClassifier(input_size=10, hidden_size=64, num_classes=num_classes).to(device)
X_device = X.to(device)
y_device = y_true.to(device)

with torch.no_grad():
    teacher_logits = teacher(X_device)
    teacher_probs = F.softmax(teacher_logits, dim=1)

# Show example: compare hard vs soft labels
print("Example: Hard vs Soft Labels")
print("-" * 60)
print("Sample index: 0")
print(f"\nTrue class (hard label): {y_true[0].item()}")
print(f"Hard label (one-hot): {F.one_hot(y_true[0], num_classes=num_classes).cpu().numpy()}")

print(f"\nTeacher logits: {teacher_logits[0].cpu().detach().numpy()}")
print(f"Teacher probabilities (soft label):")
for i, prob in enumerate(teacher_probs[0].cpu().detach().numpy()):
    print(f"  Class {i}: {prob:.4f}")

print(f"\nKey observation:")
print(f"  Hard label says class {y_true[0]} is 100%, others are 0%")
print(f"  Soft label reveals semantic structure:")
print(f"    - Class {y_true[0]} is {teacher_probs[0, y_true[0]].item():.1%} likely")
print(f"    - Class {teacher_probs[0].argmax(dim=0).item()} is most likely")
print(f"    - Small probabilities show similarity to other classes")


## Level 2: Knowledge Distillation Loss with Temperature Scaling

Implement temperature-scaled softmax for soft target generation.

In [ ]:
def distillation_loss(
    student_logits: torch.Tensor,
    teacher_logits: torch.Tensor,
    labels: torch.Tensor,
    temperature: float = 4.0,
    alpha: float = 0.7
) -> torch.Tensor:
    """
    Combined distillation loss: soft targets from teacher + hard targets from labels.
    
    Args:
        student_logits: Output logits from student model (B, C)
        teacher_logits: Output logits from teacher model (B, C), no gradient
        labels: Ground truth labels (B,)
        temperature: Temperature for softmax (higher = softer)
        alpha: Weight for distillation loss (0 to 1)
               alpha=1: pure distillation
               alpha=0: pure supervised learning
    
    Returns:
        loss: Scalar loss
    """
    # Soft target loss: KL divergence between teacher and student probabilities
    soft_targets = F.softmax(teacher_logits / temperature, dim=1)
    student_log_probs = F.log_softmax(student_logits / temperature, dim=1)
    
    # KL divergence loss
    loss_distill = F.kl_div(student_log_probs, soft_targets, reduction='batchmean')
    
    # Hard target loss: standard cross-entropy
    loss_hard = F.cross_entropy(student_logits, labels)
    
    # Combined loss with scaling
    # T^2 factor ensures gradients are scaled appropriately
    loss = alpha * (temperature ** 2) * loss_distill + (1 - alpha) * loss_hard
    
    return loss

def standard_loss(student_logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """Standard cross-entropy loss (no distillation)."""
    return F.cross_entropy(student_logits, labels)

# Train three models to compare:
# 1. Teacher model (well-trained)
# 2. Student from scratch (no distillation)
# 3. Student with distillation

# Create synthetic dataset
n_samples = 1000
X_train = torch.randn(n_samples, 10)
y_train = torch.randint(0, 4, (n_samples,))
dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Teacher model
print("Training teacher model...")
teacher = SimpleClassifier().to(device)
optimizer_teacher = optim.Adam(teacher.parameters(), lr=0.001)

teacher_losses = []
for epoch in range(10):
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        logits = teacher(X_batch)
        loss = standard_loss(logits, y_batch)
        
        optimizer_teacher.zero_grad()
        loss.backward()
        optimizer_teacher.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    teacher_losses.append(avg_loss)
    if (epoch + 1) % 3 == 0:
        print(f"  Epoch {epoch+1}/10 - Loss: {avg_loss:.4f}")

teacher.eval()
print(f"Teacher training complete! Final loss: {teacher_losses[-1]:.4f}
")

# Student from scratch (baseline)
print("Training student from scratch (no distillation)...")
student_scratch = SimpleClassifier().to(device)
optimizer_scratch = optim.Adam(student_scratch.parameters(), lr=0.001)

scratch_losses = []
for epoch in range(10):
    epoch_loss = 0
    student_scratch.train()
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        logits = student_scratch(X_batch)
        loss = standard_loss(logits, y_batch)
        
        optimizer_scratch.zero_grad()
        loss.backward()
        optimizer_scratch.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    scratch_losses.append(avg_loss)

print(f"Student (scratch) training complete! Final loss: {scratch_losses[-1]:.4f}
")

# Student with distillation
print("Training student with knowledge distillation...")
student_distill = SimpleClassifier().to(device)
optimizer_distill = optim.Adam(student_distill.parameters(), lr=0.001)

distill_losses = []
distill_hard_losses = []

for epoch in range(10):
    epoch_loss = 0
    epoch_hard_loss = 0
    student_distill.train()
    teacher.eval()
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        # Get student logits
        student_logits = student_distill(X_batch)
        
        # Get teacher logits (no gradient)
        with torch.no_grad():
            teacher_logits = teacher(X_batch)
        
        # Distillation loss
        loss = distillation_loss(
            student_logits, 
            teacher_logits, 
            y_batch,
            temperature=4.0,
            alpha=0.7  # 70% distillation, 30% hard labels
        )
        
        optimizer_distill.zero_grad()
        loss.backward()
        optimizer_distill.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    distill_losses.append(avg_loss)

print(f"Student (distilled) training complete! Final loss: {distill_losses[-1]:.4f}
")


In [ ]:
# Evaluate on test set
X_test = torch.randn(200, 10)
y_test = torch.randint(0, 4, (200,))

X_test_device = X_test.to(device)
y_test_device = y_test.to(device)

def compute_accuracy(model, X, y):
    """Compute classification accuracy."""
    model.eval()
    with torch.no_grad():
        logits = model(X)
        predictions = logits.argmax(dim=1)
        accuracy = (predictions == y).float().mean().item()
    return accuracy

acc_teacher = compute_accuracy(teacher, X_test_device, y_test_device)
acc_scratch = compute_accuracy(student_scratch, X_test_device, y_test_device)
acc_distill = compute_accuracy(student_distill, X_test_device, y_test_device)

print("Evaluation Results:")
print("-" * 60)
print(f"Teacher accuracy: {acc_teacher:.4f}")
print(f"Student (scratch) accuracy: {acc_scratch:.4f}")
print(f"Student (distilled) accuracy: {acc_distill:.4f}")
print("-" * 60)

# Calculate accuracy recovery
if acc_teacher > acc_scratch:
    accuracy_gap = acc_teacher - acc_scratch
    improvement = acc_distill - acc_scratch
    recovery_pct = improvement / accuracy_gap * 100 if accuracy_gap > 0 else 0
    
    print(f"\nAccuracy improvement from distillation:")
    print(f"  Gap to recover: {accuracy_gap:.4f}")
    print(f"  Distillation recovery: {improvement:.4f}")
    print(f"  Recovery percentage: {recovery_pct:.1f}%")

# Model size comparison
teacher_params = sum(p.numel() for p in teacher.parameters())
student_params = sum(p.numel() for p in student_distill.parameters())
compression_ratio = teacher_params / student_params

print(f"\nModel size comparison:")
print(f"  Teacher: {teacher_params:,} parameters")
print(f"  Student: {student_params:,} parameters")
print(f"  Compression: {compression_ratio:.1f}x")


## Real-World Example 1: Temperature Effect on Soft Targets

Visualize how temperature affects information in soft targets.

In [ ]:
# Show effect of different temperatures on soft targets
temperatures = [1.0, 2.0, 4.0, 8.0, 16.0]

with torch.no_grad():
    sample_logits = teacher(X_test_device[:1])  # (1, 4)
    print("Temperature Effect on Soft Targets:")
    print("-" * 70)
    print("Temperature | P(class0) | P(class1) | P(class2) | P(class3) | Entropy")
    print("-" * 70)
    
    entropies = []
    soft_probs_list = []
    
    for T in temperatures:
        soft_probs = F.softmax(sample_logits / T, dim=1)[0]
        soft_probs_list.append(soft_probs.cpu().numpy())
        
        # Compute entropy
        entropy = -(soft_probs * torch.log(soft_probs + 1e-8)).sum().item()
        entropies.append(entropy)
        
        print(f"{T:11.1f} | {soft_probs[0]:9.4f} | {soft_probs[1]:9.4f} | "
              f"{soft_probs[2]:9.4f} | {soft_probs[3]:9.4f} | {entropy:7.4f}")

print("-" * 70)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Soft probabilities vs temperature
x = np.arange(4)
width = 0.15

for i, T in enumerate(temperatures):
    offset = (i - 2) * width
    axes[0].bar(x + offset, soft_probs_list[i], width, label=f'T={T}', alpha=0.8)

axes[0].set_xlabel('Class', fontsize=11)
axes[0].set_ylabel('Probability', fontsize=11)
axes[0].set_title('Soft Target Probabilities vs Temperature', fontsize=12, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'Class {i}' for i in range(4)])
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Entropy vs temperature
axes[1].plot(temperatures, entropies, marker='o', linewidth=2, markersize=8, color='blue')
axes[1].axhline(y=np.log(4), color='r', linestyle='--', label='Max entropy (uniform)')
axes[1].set_xlabel('Temperature', fontsize=11)
axes[1].set_ylabel('Entropy of soft targets', fontsize=11)
axes[1].set_title('Information in Soft Targets', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/temperature_effect.png', dpi=100, bbox_inches='tight')
print("\nVisualization saved to /tmp/temperature_effect.png")

print("\nKey insight: Higher T reveals more information about wrong answers")
print("(soft targets become smoother, less peaked)")


## Real-World Example 2: Multi-Teacher Distillation

Combine knowledge from multiple teachers for richer learning signal.

In [ ]:
def multi_teacher_distillation_loss(
    student_logits: torch.Tensor,
    teacher_logits_list: List[torch.Tensor],
    labels: torch.Tensor,
    temperature: float = 4.0,
    alpha: float = 0.7
) -> torch.Tensor:
    """
    Distillation loss from multiple teachers (average their soft targets).
    
    Args:
        student_logits: (B, C)
        teacher_logits_list: List of (B, C) teacher logits
        labels: (B,)
        temperature: Temperature for softmax
        alpha: Weight for distillation
    
    Returns:
        loss: Scalar loss
    """
    # Average soft targets from all teachers
    avg_soft_targets = None
    for teacher_logits in teacher_logits_list:
        soft_targets = F.softmax(teacher_logits / temperature, dim=1)
        if avg_soft_targets is None:
            avg_soft_targets = soft_targets
        else:
            avg_soft_targets += soft_targets
    
    avg_soft_targets = avg_soft_targets / len(teacher_logits_list)
    
    # Student soft predictions
    student_log_probs = F.log_softmax(student_logits / temperature, dim=1)
    
    # KL divergence
    loss_distill = F.kl_div(student_log_probs, avg_soft_targets, reduction='batchmean')
    
    # Hard loss
    loss_hard = F.cross_entropy(student_logits, labels)
    
    # Combined
    loss = alpha * (temperature ** 2) * loss_distill + (1 - alpha) * loss_hard
    return loss

# Create multiple teachers with different random seeds
print("Training 3 teacher models (different seeds)...")
teachers_multi = []

for seed in [42, 123, 456]:
    torch.manual_seed(seed)
    teacher_i = SimpleClassifier().to(device)
    optimizer_i = optim.Adam(teacher_i.parameters(), lr=0.001)
    
    # Quick training
    for epoch in range(10):
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = teacher_i(X_batch)
            loss = F.cross_entropy(logits, y_batch)
            optimizer_i.zero_grad()
            loss.backward()
            optimizer_i.step()
    
    teacher_i.eval()
    teachers_multi.append(teacher_i)
    acc = compute_accuracy(teacher_i, X_test_device, y_test_device)
    print(f"  Teacher {len(teachers_multi)} trained. Accuracy: {acc:.4f}")

# Train student from multiple teachers
print("\nTraining student from multiple teachers...")
student_multi = SimpleClassifier().to(device)
optimizer_multi = optim.Adam(student_multi.parameters(), lr=0.001)

multi_losses = []
for epoch in range(10):
    epoch_loss = 0
    student_multi.train()
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        student_logits = student_multi(X_batch)
        
        # Get all teacher logits
        teacher_logits_list = []
        with torch.no_grad():
            for teacher_i in teachers_multi:
                teacher_logits = teacher_i(X_batch)
                teacher_logits_list.append(teacher_logits)
        
        # Multi-teacher distillation loss
        loss = multi_teacher_distillation_loss(
            student_logits,
            teacher_logits_list,
            y_batch,
            temperature=4.0,
            alpha=0.7
        )
        
        optimizer_multi.zero_grad()
        loss.backward()
        optimizer_multi.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    multi_losses.append(avg_loss)

acc_multi = compute_accuracy(student_multi, X_test_device, y_test_device)
print(f"Multi-teacher student accuracy: {acc_multi:.4f}")

# Compare all methods
print("\nComparison: All Distillation Methods")
print("-" * 60)
print(f"Single teacher accuracy: {acc_teacher:.4f}")
print(f"Student (scratch): {acc_scratch:.4f}")
print(f"Student (single teacher): {acc_distill:.4f}")
print(f"Student (multi-teacher): {acc_multi:.4f}")
print("-" * 60)

print(f"\nMulti-teacher benefit: +{(acc_multi - acc_distill)*100:.2f}% over single teacher")


## Training Curves Comparison

In [ ]:
# Create comprehensive comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Training loss curves
epochs = range(1, 11)
axes[0, 0].plot(epochs, teacher_losses, marker='o', label='Teacher', linewidth=2)
axes[0, 0].plot(epochs, scratch_losses, marker='s', label='Student (scratch)', linewidth=2)
axes[0, 0].plot(epochs, distill_losses, marker='^', label='Student (distilled)', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=11)
axes[0, 0].set_ylabel('Loss', fontsize=11)
axes[0, 0].set_title('Training Loss Comparison', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Accuracy bar chart
methods = ['Teacher', 'Student
(scratch)', 'Student
(distilled)', 'Student
(multi-teacher)']
accuracies = [acc_teacher, acc_scratch, acc_distill, acc_multi]
colors = ['blue', 'orange', 'green', 'red']

bars = axes[0, 1].bar(methods, accuracies, color=colors, alpha=0.7)
axes[0, 1].set_ylabel('Accuracy', fontsize=11)
axes[0, 1].set_title('Final Accuracy Comparison', fontsize=12, fontweight='bold')
axes[0, 1].set_ylim([0, 1.0])
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                   f'{acc:.4f}', ha='center', va='bottom', fontsize=10)

# Plot 3: Temperature sweep (from earlier)
axes[1, 0].plot(temperatures, entropies, marker='o', linewidth=2, markersize=8)
axes[1, 0].axhline(y=np.log(4), color='r', linestyle='--', label='Max entropy')
axes[1, 0].set_xlabel('Temperature', fontsize=11)
axes[1, 0].set_ylabel('Entropy of soft targets', fontsize=11)
axes[1, 0].set_title('Temperature Effect on Information', fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Compression vs accuracy
compression_ratios = [1, compression_ratio, compression_ratio, compression_ratio]
accuracy_drops = [0, acc_teacher - acc_scratch, 
                 acc_teacher - acc_distill, acc_teacher - acc_multi]
labels_scatter = ['Teacher', 'Student
(scratch)', 'Student
(distilled)', 'Student
(multi-teacher)']

axes[1, 1].scatter(compression_ratios, accuracy_drops, s=300, alpha=0.7, c=colors)
for i, label in enumerate(labels_scatter):
    axes[1, 1].annotate(label, (compression_ratios[i], accuracy_drops[i]),
                       xytext=(5, 5), textcoords='offset points', fontsize=9)

axes[1, 1].set_xlabel('Compression Ratio (teacher / student)', fontsize=11)
axes[1, 1].set_ylabel('Accuracy Drop from Teacher', fontsize=11)
axes[1, 1].set_title('Compression vs Accuracy Trade-off', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.savefig('/tmp/knowledge_distillation_analysis.png', dpi=100, bbox_inches='tight')
print("Visualization saved to /tmp/knowledge_distillation_analysis.png")


## Key Takeaways

**Core Insight:** Soft targets from teachers encode "dark knowledge" about decision boundaries.

**Why Distillation Works:**
1. **Smoother targets:** Soft probabilities provide gradient signal even for correct predictions
2. **Semantic structure:** Small probabilities for wrong answers reveal similarity
3. **Better regularization:** Smooth targets prevent overfitting and sharpen decision boundaries
4. **Curriculum learning:** Easy examples (high confidence) serve as learning targets first

**Temperature Role:**
| T | Effect | Information |
|---|--------|-------------|
| 1.0 | Sharp targets (peaked) | Low (only top class) |
| 4.0 | Moderate smoothness | High (reveals structure) |
| 20.0 | Flat targets (uniform) | None (all classes equal) |

**Distillation Strategy:**
- Single teacher: For most cases, 1-2% accuracy drop
- Multi-teacher: Average 2-3 teachers for ensemble effect
- Combined loss: alpha=0.7 distillation + 0.3 hard labels (ground truth)

**When to Use:**
- **Distillation:** Need 10-100x compression with < 2% accuracy loss
- **Pruning:** Need structured sparsity (model architecture changes)
- **Quantization:** Need speed over memory (2-3x reduction, not 10-100x)

**Real-World Results:**
- BERT 340M -> BERT 110M with distillation: 99% accuracy recovery
- BERT 110M -> BERT 4M with distillation: 90%+ accuracy recovery
- GPT models: distilled to 10% model size, 95%+ quality


## Try It Yourself

1. **Hyperparameter Tuning:** Try different temperatures (T=1, 2, 4, 8, 16) and alpha values (0.3, 0.5, 0.7, 0.9). Which combination gives best accuracy?

2. **Student Architecture Variations:** How small can the student be? Try hidden_size=16, 32, 64. At what point does distillation not help?

3. **Curriculum Distillation:** Start with high alpha (more hard labels), gradually decrease to emphasize soft targets. Does this improve convergence?

4. **Cross-Model Distillation:** Train teacher and student on different data. How does distribution mismatch affect distillation effectiveness?

5. **Layer-wise Distillation:** Distill intermediate representations (not just final logits). Does this improve accuracy further?

6. **Distillation + Quantization:** Combine knowledge distillation with INT8 quantization. Can small quantized students match large float16 teachers?
